# Phase B0 — label-sensitive datasets and 60-s windows

Create **T30, T60, S3, S5** from the processed session CSVs, using the existing SQI, segmentation and quasi-stationarity cores. P0 is not regenerated. No NTSA method is run.

Each rule saves 20 filtered session CSVs and 20 candidate-window tables, including empty tables when needed. The existing `PPG processed` values are used without refiltering. Labels, original timestamps and disconnected intervals are preserved. All experiment orchestration is inline in this notebook; existing core modules remain unchanged.

In [1]:
from pathlib import Path
import hashlib
import inspect
import json
import re
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "phase1/src/segmentation").is_dir())
PHASE1_DIR = PROJECT_ROOT / "phase1"
INPUT_DIR = PHASE1_DIR / "data_processed" / "dhdata"
OUTPUT_DIR = PHASE1_DIR / "results" / "data_label_sensitive"
sys.path.insert(0, str(PHASE1_DIR / "src"))
from preprocessing.sqi import detect_motion_artifacts
from segmentation.segmentation import segment_session, validate_quasi_stationarity

RULES = ("T30", "T60", "S3", "S5")
LABEL_MAP = {0: "Awake", 1: "Drowsy"}
EXPECTED_SESSIONS = 20
WINDOW_S = 60
# Fixed primary settings: notebook/data_sqi.ipynb and data_segmentation.ipynb.
SQI_WINDOW_S = 5.0
SQI_THRESHOLD = 4.5
STATIONARITY_SUBWINDOW_S = 10.0
STATIONARITY_THRESHOLD = 0.5
GAP_FACTOR = inspect.signature(segment_session).parameters["gap_factor"].default
REQUIRED_COLUMNS = ["Time (s)", "IR Value raw", "PPG processed", "Label"]
files = sorted(INPUT_DIR.glob("*.csv"), key=lambda p: [int(v) if v.isdigit() else v for v in re.split(r"(\d+)", p.name)])
assert len(files) == EXPECTED_SESSIONS, f"Expected 20 session files; found {len(files)}"
print(f"Input: {INPUT_DIR}\nOutput: {OUTPUT_DIR}")

Input: /home/vutu0809/Desktop/NTSA_Foundation/phase1/data_processed/dhdata
Output: /home/vutu0809/Desktop/NTSA_Foundation/phase1/results/data_label_sensitive


## Frozen rule and QC conventions

- A label segment ends at a state change or `dt > 1.5 × session median_dt`. Segment duration follows Phase A sample support: last timestamp minus first timestamp plus the last sample's support; support across a gap and at session end is `median_dt`.
- A transition is the first sample carrying the new state. T30/T60 exclude distances **≤30/60 s**, taking the union of overlapping regions. S3/S5 keep original segments lasting **≥180/300 s**, respectively.
- SQI is recomputed using `detect_motion_artifacts` on the **original processed session**, with the primary pipeline's session-wide median/MAD normalization and 5-s grid. Its mask is then selected by original row index. It is not recalibrated separately per retained interval or per 60-s window. This preserves the existing primary SQI calibration and avoids joining filtered signal fragments. SQI's existing treatment of an incomplete 5-s tail is unchanged.
- Each retained contiguous interval is passed separately to `segment_session`. Candidate enumeration uses an all-False SQI mask so rejected candidates remain auditable; a second call with the actual SQI mask verifies the pass decisions against the core. Both calls use the same non-overlapping sample grid and `round(60 × session fs)` samples.
- Quasi-stationarity is evaluated on every candidate using the existing core. `stationarity_pass` and final inclusion refer to **processed** PPG, matching the primary retention pipeline; raw-signal scores are also recorded. `analysis_included = sqi_pass AND stationarity_pass`.
- `stationarity_pass_windows` counts stationarity passes among all candidates; `stationarity_pass_after_sqi_windows` counts passes after SQI and equals final included windows. No mean-stability metric or single SQI score is invented: the SQI core returns a boolean sample mask, so artifact sample count/fraction is recorded instead.

Filtered files preserve the source signal columns and add original row, segment and retained-interval identifiers. Window indices are **zero-based, half-open** in the corresponding filtered CSV; source row bounds are included for tracing back to the original session.

In [2]:
WINDOW_COLUMNS = [
    "subject", "session", "source_file", "rule", "window_id", "state", "label", "segment_id", "retained_interval_id",
    "window_start_s", "window_end_s", "duration_s", "observed_span_s", "n_samples", "fs",
    "window_start_index", "window_end_index", "source_start_index", "source_end_index", "signal_file",
    "sqi_pass", "sqi_artifact_samples", "sqi_artifact_fraction", "stationarity_pass", "variance_stability_metric",
    "stationarity_pass_raw", "variance_stability_metric_raw", "analysis_included", "exclusion_reason",
]
INTERVAL_COLUMNS = ["session", "rule", "segment_id", "retained_interval_id", "state", "source_start_index", "source_end_index",
                    "filtered_start_index", "filtered_end_index", "start_s", "end_s", "duration_s", "n_samples"]

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def true_runs(mask, boundaries):
    starts = mask & ~np.r_[False, mask[:-1]]
    ends = mask & ~np.r_[mask[1:], False]
    starts |= mask & boundaries
    ends |= mask & np.r_[boundaries[1:], False]
    return list(zip(np.flatnonzero(starts), np.flatnonzero(ends) + 1))

def label_rule_masks(time, labels):
    """Phase A rules, evaluated at full source-CSV numeric precision."""
    dt = np.diff(time)
    median_dt = float(np.median(dt))
    gaps = dt > GAP_FACTOR * median_dt
    boundaries = np.r_[True, gaps | (labels[1:] != labels[:-1])]
    support = np.r_[np.where(gaps, median_dt, dt), median_dt]
    segment_id = np.cumsum(boundaries)
    segment_duration = np.zeros(len(time))
    for a, b in true_runs(np.ones(len(time), bool), boundaries):
        segment_duration[a:b] = time[b-1] - time[a] + support[b-1]
    transitions = time[1:][labels[1:] != labels[:-1]]
    if len(transitions):
        pos = np.searchsorted(transitions, time)
        distance = np.minimum(np.abs(time-transitions[np.maximum(pos-1, 0)]),
                              np.abs(time-transitions[np.minimum(pos, len(transitions)-1)]))
    else:
        distance = np.full(len(time), np.nan)
    masks = {"T30": ~(distance <= 30), "T60": ~(distance <= 60),
             "S3": segment_duration >= 180, "S5": segment_duration >= 300}
    return masks, boundaries, segment_id, support, median_dt

def read_source(path):
    frame = pd.read_csv(path)
    assert set(REQUIRED_COLUMNS).issubset(frame.columns), f"Missing columns: {path.name}"
    for column in REQUIRED_COLUMNS:
        values = pd.to_numeric(frame[column], errors="raise").to_numpy(dtype=float)
        assert np.isfinite(values).all(), f"Nonfinite {column}: {path.name}"
    assert frame.Label.isin(LABEL_MAP).all(), f"Invalid labels: {path.name}"
    assert len(frame) >= 2 and np.all(np.diff(frame["Time (s)"]) > 0), f"Invalid timestamps: {path.name}"
    reserved = {"source_row_index", "segment_id", "retained_interval_id", "sqi_artifact", "rule"}
    assert not reserved.intersection(frame.columns), "Source already contains reserved output columns"
    return frame


def prepare_session(path):
    frame = read_source(path)
    t = frame["Time (s)"].to_numpy(dtype=float)
    labels = frame.Label.to_numpy(dtype=int)
    masks, boundaries, segment_ids, support, median_dt = label_rule_masks(t, labels)
    fs = 1.0 / median_dt
    artifact = detect_motion_artifacts(frame["PPG processed"].to_numpy(dtype=float), fs,
                                       window_s=SQI_WINDOW_S, threshold=SQI_THRESHOLD)
    session = re.search(r"(\d+)$", path.stem).group(1).zfill(2)
    subject_match = re.search(r"(?:subject|participant|sub)[_-]?(\d+)", path.stem, re.I)
    subject = subject_match.group(1) if subject_match else None
    output = []
    for rule in RULES:
        root = OUTPUT_DIR / rule
        (root / "filtered_sessions").mkdir(parents=True, exist_ok=True)
        (root / "windows_60s").mkdir(parents=True, exist_ok=True)
        keep = masks[rule]
        indices = np.flatnonzero(keep)
        filtered = frame.iloc[indices].copy()
        filtered["source_row_index"] = indices
        filtered["segment_id"] = segment_ids[indices]
        filtered["retained_interval_id"] = 0
        filtered["sqi_artifact"] = artifact[indices]
        filtered["rule"] = rule
        intervals, rows = [], []
        offset = 0
        expected_samples = int(round(WINDOW_S * fs))
        for interval_id, (a, b) in enumerate(true_runs(keep, boundaries), start=1):
            length = b-a
            filtered.iloc[offset:offset+length, filtered.columns.get_loc("retained_interval_id")] = interval_id
            intervals.append(dict(session=session, rule=rule, segment_id=int(segment_ids[a]), retained_interval_id=interval_id,
                state=LABEL_MAP[labels[a]], source_start_index=int(a), source_end_index=int(b),
                filtered_start_index=offset, filtered_end_index=offset+length,
                start_s=t[a], end_s=t[b-1], duration_s=float(t[b-1]-t[a]+support[b-1]), n_samples=int(length)))
            args = dict(time_s=t[a:b], ppg_raw=frame["IR Value raw"].to_numpy()[a:b],
                        ppg_processed=frame["PPG processed"].to_numpy()[a:b], labels=labels[a:b],
                        fs=fs, session_id=session, window_sizes=(WINDOW_S,), gap_factor=GAP_FACTOR)
            candidates = segment_session(**args, sqi_mask=np.zeros(length, dtype=bool))[WINDOW_S]
            core_pass = segment_session(**args, sqi_mask=artifact[a:b])[WINDOW_S]
            core_pass_starts = {w["start_time"] for w in core_pass}
            assert len(candidates) == length // expected_samples, "Unexpected structural candidate rejection"
            for index, window in enumerate(candidates):
                local_start = index * expected_samples
                source_start = a + local_start
                source_end = source_start + expected_samples
                window_artifact = artifact[source_start:source_end]
                sqi_pass = bool(not window_artifact.any())
                assert sqi_pass == (window["start_time"] in core_pass_starts)
                score, stationarity_pass = validate_quasi_stationarity(window, signal_key="ppg_processed",
                    subwindow_s=STATIONARITY_SUBWINDOW_S, threshold=STATIONARITY_THRESHOLD)
                raw_score, raw_pass = validate_quasi_stationarity(window, signal_key="ppg_raw",
                    subwindow_s=STATIONARITY_SUBWINDOW_S, threshold=STATIONARITY_THRESHOLD)
                included = bool(sqi_pass and stationarity_pass)
                reasons = ([] if sqi_pass else ["SQI"]) + ([] if stationarity_pass else ["processed_stationarity"])
                assert source_end <= b and keep[source_start:source_end].all()
                assert np.all(labels[source_start:source_end] == window["label"])
                assert not np.any(boundaries[source_start+1:source_end])
                assert np.all(np.diff(window["time"]) <= GAP_FACTOR / fs)
                assert np.array_equal(window["ppg_processed"], frame["PPG processed"].to_numpy()[source_start:source_end])
                rows.append(dict(subject=subject, session=session, source_file=path.name, rule=rule,
                    window_id=f"{rule}_{session}_{len(rows)+1:05d}", state=LABEL_MAP[window["label"]], label=window["label"],
                    segment_id=int(segment_ids[a]), retained_interval_id=interval_id,
                    window_start_s=window["start_time"], window_end_s=window["end_time"], duration_s=WINDOW_S,
                    observed_span_s=window["end_time"]-window["start_time"], n_samples=expected_samples, fs=fs,
                    window_start_index=offset+local_start, window_end_index=offset+local_start+expected_samples,
                    source_start_index=int(source_start), source_end_index=int(source_end),
                    signal_file=f"{rule}/filtered_sessions/{path.name}", sqi_pass=sqi_pass,
                    sqi_artifact_samples=int(window_artifact.sum()), sqi_artifact_fraction=float(window_artifact.mean()),
                    stationarity_pass=stationarity_pass, variance_stability_metric=score,
                    stationarity_pass_raw=raw_pass, variance_stability_metric_raw=raw_score,
                    analysis_included=included, exclusion_reason=";".join(reasons)))
            offset += length
        assert offset == len(filtered)
        window_table = pd.DataFrame(rows, columns=WINDOW_COLUMNS)
        interval_table = pd.DataFrame(intervals, columns=INTERVAL_COLUMNS)
        # CSVs round-trip signal values; no vectors are embedded in window tables.
        filtered.to_csv(root / "filtered_sessions" / path.name, index=False)
        window_table.to_csv(root / "windows_60s" / path.name, index=False)
        result = dict(rule=rule, session=session, source_file=path.name, source_samples=len(frame),
                      retained_samples=len(filtered), retained_intervals=len(intervals), fs=fs)
        for state in ("Awake", "Drowsy"):
            group = window_table[window_table.state == state]
            slug = state.lower()
            result.update({f"{slug}_candidate_windows": len(group),
                f"{slug}_sqi_pass_windows": int(group.sqi_pass.sum()),
                f"{slug}_stationarity_pass_windows": int(group.stationarity_pass.sum()),
                f"{slug}_stationarity_pass_after_sqi_windows": int(group.analysis_included.sum()),
                f"{slug}_included_windows": int(group.analysis_included.sum())})
        result["paired"] = bool(result["awake_included_windows"] and result["drowsy_included_windows"])
        output.append((result, interval_table))
    return output

## Boundary checks before dataset creation

These small checks verify inclusive transition exclusions, overlapping exclusions, sustained-segment thresholds, and separation across acquisition gaps. The existing SQI/segmentation/stationarity methods themselves are reused unchanged.

In [3]:
t = np.arange(240., dtype=float)
labels = np.r_[np.zeros(80, int), np.ones(40, int), np.zeros(120, int)]
masks, boundaries, _, _, _ = label_rule_masks(t, labels)
assert list(true_runs(masks["T30"], boundaries)) == [(0, 50), (151, 240)]
assert masks["T30"].sum() == 139  # union [50, 150], inclusive
masks, boundaries, _, _, _ = label_rule_masks(np.r_[np.arange(40.), np.arange(100., 140.)], np.zeros(80, int))
assert list(true_runs(masks["T30"], boundaries)) == [(0, 40), (40, 80)]
assert not masks["S3"].any()  # same-state gap must not contribute missing time
masks, _, _, _, _ = label_rule_masks(np.arange(180.), np.zeros(180, int))
assert masks["S3"].all() and not masks["S5"].any()
assert masks["T60"].all()  # no transitions
print("Rule boundary checks passed.")

Rule boundary checks passed.


## Run the four sensitivity datasets

Processing is sequential by session. A failed structural assertion stops the run rather than silently dropping a session. Stationarity and SQI flags remain in each window table, including rejected candidates. Original session SHA-256 hashes and core hashes are recorded for reproducibility.

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = [{"source_file": p.name, "sha256": sha256(p), "n_bytes": p.stat().st_size} for p in files]
summary_rows, interval_frames = [], []
for path in files:
    for result, intervals in prepare_session(path):
        summary_rows.append(result)
        interval_frames.append(intervals)
    print(f"Completed {path.name}: T30 / T60 / S3 / S5")
per_session_summary = pd.DataFrame(summary_rows)
retained_intervals = pd.concat(interval_frames, ignore_index=True)
master_rows = []
for rule in RULES:
    group = per_session_summary[per_session_summary.rule == rule]
    row = dict(rule=rule, n_sessions=len(group), paired_sessions=int(group.paired.sum()))
    for column in group.columns:
        if column.endswith("_windows"):
            row[column] = int(group[column].sum())
    for state in ("awake", "drowsy"):
        before = row[f"{state}_candidate_windows"]
        row[f"{state}_inclusion_rate"] = row[f"{state}_included_windows"] / before if before else np.nan
    master_rows.append(row)
    group.to_csv(OUTPUT_DIR / rule / "per_session_summary.csv", index=False)
    pd.DataFrame([row]).to_csv(OUTPUT_DIR / rule / "summary.csv", index=False)
    retained_intervals[retained_intervals.rule == rule].to_csv(OUTPUT_DIR / rule / "retained_intervals.csv", index=False)
master_summary = pd.DataFrame(master_rows)
master_summary.to_csv(OUTPUT_DIR / "master_summary.csv", index=False)
per_session_summary.to_csv(OUTPUT_DIR / "per_session_summary.csv", index=False)
pd.DataFrame(manifest).to_csv(OUTPUT_DIR / "input_manifest.csv", index=False)
configuration = dict(rules=RULES, window_s=WINDOW_S, overlap_s=0,
    sqi_window_s=SQI_WINDOW_S, sqi_threshold=SQI_THRESHOLD,
    sqi_scope="Original processed session, primary session-wide calibration and grid; mask selected by original row",
    stationarity_signal="PPG processed", stationarity_subwindow_s=STATIONARITY_SUBWINDOW_S,
    stationarity_threshold=STATIONARITY_THRESHOLD, gap_factor=GAP_FACTOR,
    transition_exclusion_inclusive=True, segment_threshold_s={"S3":180,"S5":300},
    index_convention="zero-based, half-open; window indices reference filtered signal CSV",
    filtering="Existing PPG processed unchanged; no refiltering", p0_regenerated=False, ntsa_computed=False,
    parameter_sources=["notebook/data_sqi.ipynb", "notebook/data_segmentation.ipynb"],
    numpy_version=np.__version__, pandas_version=pd.__version__,
    core_sha256={str(p.relative_to(PHASE1_DIR)):sha256(p) for p in [PHASE1_DIR/"src/preprocessing/sqi.py", PHASE1_DIR/"src/segmentation/segmentation.py"]})
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(configuration, indent=2), encoding="utf-8")

Completed sample_1.csv: T30 / T60 / S3 / S5


Completed sample_4.csv: T30 / T60 / S3 / S5


Completed sample_5.csv: T30 / T60 / S3 / S5


Completed sample_6.csv: T30 / T60 / S3 / S5


Completed sample_7.csv: T30 / T60 / S3 / S5


...

Completed sample_21.csv: T30 / T60 / S3 / S5


Completed sample_22.csv: T30 / T60 / S3 / S5


Completed sample_23.csv: T30 / T60 / S3 / S5


Completed sample_25.csv: T30 / T60 / S3 / S5


1099

## Validate saved files and direct signal access

Reload every filtered session and window table. Check exact signal/label preservation against selected source rows, interval membership, continuous source-row indices, unmixed labels, acquisition-gap boundaries, QC inclusion logic, unique non-overlapping windows and all 20 session files for every rule. Input hashes are checked again after processing.

In [5]:
validation_rows = []
for path in files:
    source = read_source(path)
    source_time = source["Time (s)"].to_numpy(dtype=float)
    masks, boundaries, _, _, median_dt = label_rule_masks(source_time, source.Label.to_numpy(dtype=int))
    assert sha256(path) == next(r["sha256"] for r in manifest if r["source_file"] == path.name)
    for rule in RULES:
        root = OUTPUT_DIR / rule
        signal = pd.read_csv(root / "filtered_sessions" / path.name, float_precision="round_trip")
        windows = pd.read_csv(root / "windows_60s" / path.name, float_precision="round_trip")
        ids = signal.source_row_index.to_numpy(dtype=int)
        assert np.array_equal(ids, np.flatnonzero(masks[rule]))
        for column in REQUIRED_COLUMNS:
            assert np.array_equal(signal[column].to_numpy(), source[column].to_numpy()[ids])
        assert windows.window_id.is_unique
        last_stop = 0
        for row in windows.itertuples():
            a, b = int(row.window_start_index), int(row.window_end_index)
            cut = signal.iloc[a:b]
            assert a >= last_stop and b-a == row.n_samples == round(WINDOW_S / median_dt)
            last_stop = b
            assert np.array_equal(cut.source_row_index, np.arange(row.source_start_index, row.source_end_index))
            assert cut.retained_interval_id.eq(row.retained_interval_id).all()
            assert cut.segment_id.eq(row.segment_id).all() and cut.Label.eq(row.label).all()
            assert masks[rule][row.source_start_index:row.source_end_index].all()
            assert not boundaries[row.source_start_index+1:row.source_end_index].any()
            assert np.all(np.diff(cut["Time (s)"]) <= GAP_FACTOR * median_dt)
            assert cut["Time (s)"].iloc[0] == row.window_start_s and cut["Time (s)"].iloc[-1] == row.window_end_s
            assert row.sqi_pass == (not cut.sqi_artifact.any())
            assert row.analysis_included == (row.sqi_pass and row.stationarity_pass)
        validation_rows.append(dict(rule=rule, source_file=path.name, n_windows=len(windows),
                                    exact_signal_and_label_preservation=True, interval_boundaries_pass=True,
                                    qc_inclusion_logic_pass=True, input_unchanged=True))
for rule in RULES:
    for folder in ("filtered_sessions", "windows_60s"):
        assert {p.name for p in (OUTPUT_DIR / rule / folder).glob("*.csv")} == {p.name for p in files}
validation_summary = pd.DataFrame(validation_rows)
validation_summary.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)
print(f"Validated {len(validation_summary)} rule × session outputs, with all source hashes unchanged.")

Validated 80 rule × session outputs, with all source hashes unchanged.


## Retention after segmentation and QC

All pass counts refer to candidate windows; final inclusion is the intersection of SQI and processed-signal stationarity. A paired session has at least one included window in each state.

In [6]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(master_summary)
    display(per_session_summary)
    display(validation_summary)
report_lines = ["Phase B0 completed: T30, T60, S3, S5; P0 unchanged.",
                "20 filtered sessions and 20 window tables per rule; 60-s non-overlapping candidate windows.",
                "SQI: primary session-wide 5-s / 4.5 calibration. Stationarity: processed PPG, 10-s / 0.5."]
for row in master_summary.itertuples():
    report_lines.append(f"{row.rule}: Awake {row.awake_included_windows}/{row.awake_candidate_windows} included; "
                        f"Drowsy {row.drowsy_included_windows}/{row.drowsy_candidate_windows} included; "
                        f"paired sessions {row.paired_sessions}/{row.n_sessions}.")
report_lines += ["80 rule × session outputs passed saved-file consistency checks.",
                 "No NTSA computation, relabeling, signal refiltering or session exclusion performed."]
print("\n".join(report_lines))
(OUTPUT_DIR / "completion_summary.txt").write_text("\n".join(report_lines)+"\n", encoding="utf-8")

,rule,n_sessions,paired_sessions,awake_candidate_windows,awake_sqi_pass_windows,awake_stationarity_pass_windows,awake_stationarity_pass_after_sqi_windows,awake_included_windows,drowsy_candidate_windows,drowsy_sqi_pass_windows,drowsy_stationarity_pass_windows,drowsy_stationarity_pass_after_sqi_windows,drowsy_included_windows,awake_inclusion_rate,drowsy_inclusion_rate
0,T30,20,20,610,609,586,586,586,290,289,275,274,274,0.960656,0.944828
1,T60,20,20,574,573,551,551,551,254,253,236,236,236,0.959930,0.929134
2,S3,20,20,636,635,609,609,609,313,312,292,291,291,0.957547,0.929712
3,S5,20,19,606,605,581,581,581,286,285,266,265,265,0.958746,0.926573


,rule,session,source_file,source_samples,retained_samples,retained_intervals,fs,awake_candidate_windows,awake_sqi_pass_windows,awake_stationarity_pass_windows,awake_stationarity_pass_after_sqi_windows,awake_included_windows,drowsy_candidate_windows,drowsy_sqi_pass_windows,drowsy_stationarity_pass_windows,drowsy_stationarity_pass_after_sqi_windows,drowsy_included_windows,paired
0,T30,01,sample_1.csv,117828,96821,8,50.000000,21,21,21,21,21,8,7,7,6,6,True
1,T60,01,sample_1.csv,117828,78301,7,50.000000,18,18,18,18,18,5,4,2,2,2,True
2,S3,01,sample_1.csv,117828,92194,4,50.000000,21,21,21,21,21,8,7,6,5,5,True
3,S5,01,sample_1.csv,117828,68474,2,50.000000,17,17,17,17,17,5,4,3,2,2,True
4,T30,04,sample_4.csv,75416,69325,8,24.991878,28,28,28,28,28,14,14,14,14,14,True
5,T60,04,sample_4.csv,75416,63486,7,24.991878,27,27,27,27,27,13,13,13,13,13,True
6,S3,04,sample_4.csv,75416,71735,5,24.991878,30,30,30,30,30,16,16,16,16,16,True
7,S5,04,sample_4.csv,75416,61892,3,24.991878,27,27,27,27,27,13,13,13,13,13,True
8,T30,05,sample_5.csv,91211,83716,10,25.002500,35,34,33,33,33,17,17,16,16,16,True
9,T60,05,sample_5.csv,91211,76717,9,25.002500,34,33,32,32,32,14,14,12,12,12,True


,rule,source_file,n_windows,exact_signal_and_label_preservation,interval_boundaries_pass,qc_inclusion_logic_pass,input_unchanged
0,T30,sample_1.csv,29,True,True,True,True
1,T60,sample_1.csv,23,True,True,True,True
2,S3,sample_1.csv,29,True,True,True,True
3,S5,sample_1.csv,22,True,True,True,True
4,T30,sample_4.csv,42,True,True,True,True
5,T60,sample_4.csv,40,True,True,True,True
6,S3,sample_4.csv,46,True,True,True,True
7,S5,sample_4.csv,40,True,True,True,True
8,T30,sample_5.csv,52,True,True,True,True
9,T60,sample_5.csv,48,True,True,True,True


Phase B0 completed: T30, T60, S3, S5; P0 unchanged.
20 filtered sessions and 20 window tables per rule; 60-s non-overlapping candidate windows.
SQI: primary session-wide 5-s / 4.5 calibration. Stationarity: processed PPG, 10-s / 0.5.
T30: Awake 586/610 included; Drowsy 274/290 included; paired sessions 20/20.
T60: Awake 551/574 included; Drowsy 236/254 included; paired sessions 20/20.
S3: Awake 609/636 included; Drowsy 291/313 included; paired sessions 20/20.
S5: Awake 581/606 included; Drowsy 265/286 included; paired sessions 19/20.
80 rule × session outputs passed saved-file consistency checks.
No NTSA computation, relabeling, signal refiltering or session exclusion performed.


688

## Query a saved window for later work

Use `analysis_included == True` in a rule/session window table, open its `signal_file` relative to `OUTPUT_DIR`, then extract `PPG processed` using `.iloc[window_start_index:window_end_index]`. `window_end_s` is the last observed sample timestamp, while `duration_s = 60` is the nominal window duration from the existing core. The current notebook stops at dataset preparation.

In [7]:
# Read-only query demonstration; no downstream metric is computed.
example = None
for rule in RULES:
    for path in files:
        table = pd.read_csv(OUTPUT_DIR / rule / "windows_60s" / path.name)
        selected = table[table.analysis_included == True]
        if len(selected):
            example = selected.iloc[0]
            break
    if example is not None:
        break
if example is not None:
    signal = pd.read_csv(OUTPUT_DIR / example.signal_file, float_precision="round_trip")
    ppg_for_future_ntsa = signal["PPG processed"].iloc[int(example.window_start_index):int(example.window_end_index)].to_numpy()
    assert len(ppg_for_future_ntsa) == example.n_samples
    print(f"Query verified: {example.window_id}, {len(ppg_for_future_ntsa)} processed samples.")
else:
    print("No included window available; consult QC summaries.")

Query verified: T30_01_00001, 3000 processed samples.


## Phase B0 — tổng hợp kết quả

Đã tạo 4 bộ dữ liệu T30/T60/S3/S5. Mỗi rule gồm **20 filtered session CSV**, **20 window-level CSV**, bảng retained intervals và các bảng tổng hợp. P0 không được tạo lại.

| Rule | Awake candidate | Awake SQI-pass | Awake stationarity-pass | Awake included | Drowsy candidate | Drowsy SQI-pass | Drowsy stationarity-pass | Drowsy included | Paired sessions |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| T30 | 610 | 609 | 586 | 586 | 290 | 289 | 275 | 274 | 20/20 |
| T60 | 574 | 573 | 551 | 551 | 254 | 253 | 236 | 236 | 20/20 |
| S3 | 636 | 635 | 609 | 609 | 313 | 312 | 292 | 291 | 20/20 |
| S5 | 606 | 605 | 581 | 581 | 286 | 285 | 266 | 265 | 19/20 |

Stationarity-pass trong bảng là số pass trên toàn bộ candidate; included là giao của SQI-pass và processed-stationarity-pass. Paired session có ít nhất một included window ở mỗi trạng thái.

**Kiểm tra:** 80/80 tổ hợp rule × session đã được đọc lại và xác nhận đúng row-index, tín hiệu/label, ranh giới retained interval, gap và logic QC; SHA-256 của các file nguồn không đổi.

**Quy ước đếm:** Phase B0 dùng `round(60 × fs)` mẫu/cửa sổ theo segmentation core; Phase A dùng `floor(duration_s / 60)` để ước tính. Hai cách đếm có thể khác nhau trên timestamp không đều.

**Output:** `phase1/results/data_label_sensitive`. Tín hiệu được truy cập qua `signal_file` và các chỉ số `[window_start_index:window_end_index]` trong window table. Không tính NTSA, không relabel, không refilter, không loại session.

## Phase B0 — Final status

Status: COMPLETE

Các bộ sensitivity đã chuẩn bị:
- T30
- T60
- S3
- S5

Tất cả dataset:
- sử dụng window 60 s
- sử dụng cùng segmentation core
- sử dụng cùng SQI
- sử dụng cùng quasi-stationarity validation
- không relabel
- không refilter
- không loại session hậu nghiệm

Final included windows:

| Rule | Awake | Drowsy | Paired sessions |
|---|---:|---:|---:|
| T30 | 586 | 274 | 20/20 |
| T60 | 551 | 236 | 20/20 |
| S3 | 609 | 291 | 20/20 |
| S5 | 581 | 265 | 19/20 |

Interpretation:
- T30, T60 và S3 giữ đầy đủ cấu trúc paired 20-session.
- S5 vẫn giữ phần lớn dữ liệu nhưng chỉ còn 19 paired sessions, nên được xem là strict sensitivity.
- 80/80 rule × session combinations đã vượt qua kiểm tra consistency.
- Dataset hiện sẵn sàng cho NTSA label-sensitivity verification.

Next step:
Chạy cùng một cấu hình NTSA đã freeze trên T30/T60/S3/S5 và so sánh với kết quả P0 hiện có.